# 🔥 Proposed Model 2 — TherA + SAM2 Thermal Target Locker\n\n**Pipeline:** RGB video → TherA converts all frames → `thermal.mp4` → display thermal frame 1 → mouse click target → SAM2 mask → track all thermal frames → `output_thermal_tracking.mp4`\n\nUse a **GPU runtime** in Google Colab. TherA produces synthetic TIR imagery, not measured temperature data.\n

In [ ]:
import os, sys, subprocess, urllib.request
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Colab: Runtime → Change runtime type → T4 GPU.")

REPO = Path("/content/Target_Locker")
BRANCH = "proposed-model-2-thera-sam2"

if not REPO.exists():
    subprocess.run([
        "git","clone","--depth","1","--branch",BRANCH,
        "https://github.com/CptImtiaz/Target_Locker.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","origin",BRANCH], check=True)
    subprocess.run(["git","-C",str(REPO),"checkout",BRANCH], check=True)
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only","origin",BRANCH], check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q",
    "-r",str(REPO/"proposed_model_2"/"requirements.txt")
], check=True)

THERA = Path("/content/TherA")
if not THERA.exists():
    subprocess.run([
        "git","clone","--depth","1",
        "https://github.com/donkeymouse/TherA.git", str(THERA)
    ], check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q",
    "-r",str(THERA/"requirements.txt")
], check=True)

weights = THERA / "weights"
if not (weights / "model.pt").exists():
    subprocess.run([
        "huggingface-cli","download","donkeymouse/TherA",
        "--local-dir",str(weights)
    ], check=True)

SAM_CKPT = Path("/content/sam2.1_hiera_tiny.pt")
if not SAM_CKPT.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        SAM_CKPT
    )

sys.path.insert(0, str(REPO/"proposed_model_2"))
print("✅ Proposed Model 2 ready")
print("TherA weights:", weights)
print("SAM2 checkpoint:", SAM_CKPT)
\n

In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/content/Target_Locker")
sys.path.insert(0, str(REPO/"proposed_model_2"))
os.chdir(REPO/"proposed_model_2")

exec((REPO/"proposed_model_2"/"app_interface.py").read_text(), globals())
\n